# Imports & Constants 

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import (
    LongType, DoubleType, TimestampType, DateType, BooleanType
)
from pyspark.sql.window import Window

spark = SparkSession.builder.getOrCreate()

# Config

In [0]:
CATALOG = "devs"
SCHEMA_BRONZE = "bronze_william_lyagoba1"
SCHEMA_SILVER = "silver_william_lyagoba1"
BRONZE_TABLE = f"{CATALOG}.{SCHEMA_BRONZE}.audit_subjects"
SILVER_TABLE = f"{CATALOG}.{SCHEMA_BRONZE}.audit_subjects"
ERROR_TABLE = f"{CATALOG}.{SILVER_TABLE}.audit_subjects_cast_errors"
PARTITION_BY = "bso_organisation_code"   # partition Silver by BSO org
DATE_FMT = "yyyy-MM-dd"
TS_FMT   = "yyyy-MM-dd HH:mm:ss"

# Columns that define a unique record for deduplication
DEDUPE_KEYS = ["nhs_number"]

# Read Bronze

In [0]:
df_bronze = spark.read.format("delta").table(BRONZE_TABLE)
df_bronze.printSchema()        # Remove later
df_bronze.dtypes               # Remove later

root
 |-- nhs_number: long (nullable = true)
 |-- change_db_date_time: timestamp (nullable = true)
 |-- bso_organisation_code: string (nullable = true)
 |-- early_recall_date: date (nullable = true)
 |-- superseded_nhs_number: long (nullable = true)
 |-- removal_reason: long (nullable = true)
 |-- removal_date: date (nullable = true)
 |-- subject_status_code: string (nullable = true)
 |-- ntdd_calculation_method: string (nullable = true)
 |-- is_higher_risk: boolean (nullable = true)
 |-- preferred_language: string (nullable = true)
 |-- gp_practice_code: string (nullable = true)
 |-- next_test_due_date: date (nullable = true)
 |-- latest_invitation_date: date (nullable = true)
 |-- reason_for_ceasing_code: string (nullable = true)
 |-- higher_risk_next_test_due_date: date (nullable = true)
 |-- hr_recall_due_date: date (nullable = true)
 |-- higher_risk_referral_reason_code: string (nullable = true)
 |-- date_irradiated: date (nullable = true)
 |-- is_higher_risk_active: boolean (null

[('nhs_number', 'bigint'),
 ('change_db_date_time', 'timestamp'),
 ('bso_organisation_code', 'string'),
 ('early_recall_date', 'date'),
 ('superseded_nhs_number', 'bigint'),
 ('removal_reason', 'bigint'),
 ('removal_date', 'date'),
 ('subject_status_code', 'string'),
 ('ntdd_calculation_method', 'string'),
 ('is_higher_risk', 'boolean'),
 ('preferred_language', 'string'),
 ('gp_practice_code', 'string'),
 ('next_test_due_date', 'date'),
 ('latest_invitation_date', 'date'),
 ('reason_for_ceasing_code', 'string'),
 ('higher_risk_next_test_due_date', 'date'),
 ('hr_recall_due_date', 'date'),
 ('higher_risk_referral_reason_code', 'string'),
 ('date_irradiated', 'date'),
 ('is_higher_risk_active', 'boolean'),
 ('gene_code', 'string'),
 ('subject_postcode', 'string'),
 ('_source_file_name', 'string'),
 ('_ingestion_timestamp', 'timestamp')]

# Type Casting - Subjects

In [0]:
df_cast = (
    df_bronze

    # numeric
    .withColumn("nhs_number", F.col("nhs_number").cast(LongType()))
    .withColumn("superseded_nhs_number", F.col("superseded_nhs_number").cast(LongType()))
    .withColumn("removal_reason", F.col("removal_reason").cast(LongType()))

    # timestamps
    .withColumn("change_db_date_time", F.to_timestamp(F.col("change_db_date_time"), TS_FMT))

    # dates
    .withColumn("early_recall_date", F.to_date(F.col("early_recall_date"), DATE_FMT))
    .withColumn("removal_date", F.to_date(F.col("removal_date"), DATE_FMT))
    .withColumn("next_test_due_date", F.to_date(F.col("next_test_due_date"), DATE_FMT))
    .withColumn("latest_invitation_date", F.to_date(F.col("latest_invitation_date"), DATE_FMT))
    .withColumn("higher_risk_next_test_due_date", F.to_date(F.col("higher_risk_next_test_due_date"), DATE_FMT))
    .withColumn("hr_recall_due_date", F.to_date(F.col("hr_recall_due_date"), DATE_FMT))
    .withColumn("date_irradiated", F.to_date(F.col("date_irradiated"), DATE_FMT))

    # booleans
    .withColumn("is_higher_risk", F.col("is_higher_risk").cast(BooleanType()))
    .withColumn("is_higher_risk_active", F.col("is_higher_risk_active").cast(BooleanType()))

    # strings
    .withColumn("bso_organisation_code", F.col("bso_organisation_code").cast("string"))
    .withColumn("subject_status_code", F.col("subject_status_code").cast("string"))
    .withColumn("ntdd_calculation_method", F.col("ntdd_calculation_method").cast("string"))
    .withColumn("preferred_language", F.col("preferred_language").cast("string"))
    .withColumn("gp_practice_code", F.col("gp_practice_code").cast("string"))
    .withColumn("reason_for_ceasing_code", F.col("reason_for_ceasing_code").cast("string"))
    .withColumn("higher_risk_referral_reason_code", F.col("higher_risk_referral_reason_code").cast("string"))
    .withColumn("gene_code", F.col("gene_code").cast("string"))
    .withColumn("subject_postcode",F.col("subject_postcode").cast("string"))
)


# Quarantine rows with critical cast failures

In [0]:
CRITICAL_NOT_NULL = [
    "nhs_number",
    "change_db_date_time",
    "next_test_due_date",
    "latest_invitation_date",
]

invalid_filter = F.lit(False)
for col in CRITICAL_NOT_NULL:
    invalid_filter = invalid_filter | F.col(col).isNull()

df_invalid = df_cast.filter(invalid_filter)
invalid_count = df_invalid.count()

if invalid_count > 0:
    print(f"{invalid_count:,} rows failed critical type casting, quarantined to {ERROR_TABLE}")
    (
        df_invalid.write
        .format("delta")
        .mode("append")
        .option("mergeSchema", "true")
        .saveAsTable(ERROR_TABLE)
    )
else:
    print("No critical cast failures found.")

df_valid = df_cast.filter(~invalid_filter)

No critical cast failures found.


# Deduplication

In [0]:
window = (
    Window
    .partitionBy(*DEDUPE_KEYS)
    .orderBy(F.col("_ingestion_timestamp").desc())
)

df_silver = (
    df_valid
    .withColumn("_row_num", F.row_number().over(window))
    .filter(F.col("_row_num") == 1)
    .drop("_row_num")
)

dupe_count = df_valid.count() - df_silver.count()
print(f"Duplicates removed: {dupe_count:,}")

Duplicates removed: 56,479


# Write to Sliver table

In [0]:
(
    df_silver.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .partitionBy(PARTITION_BY)
    .saveAsTable(SILVER_TABLE)
)